In [1]:
from langchain.document_loaders import YoutubeLoader
# ! pip install youtube-transcript-api
# ! pip install pytube
# "https://www.youtube.com/watch?v=lcyHC9HLTzc",
urls = ["https://www.youtube.com/watch?v=K9mzg8ueiYA"]
save_dir = "docs/youtube/"
docs = []
for url in urls:
    loader = YoutubeLoader.from_youtube_url(url,add_video_info = True)
    docs += (loader.load())
    

In [2]:
docs

[Document(page_content="Pascal a procedural highlevel programming language famous for teaching a generation of kids from the 70s and 80s how to code it was created by Nicholas worth in the late 1960s and named after French mathematician blae Pascal it was originally based on the alol 60 language but expanded its data structuring abilities allowing developers to build Dynamic recursive data structures like trees and graphs it got its big break when it became the language of choice on the Apple 2 then Lisa and the Macintosh and eventually became the default highlevel language on nearly every PC over the years it evolved into a variety of other dialects most famously turbo Pascal brought to you by CP Creator Anders hilburg it was one of the first languages with its own full screen IDE and in 1983 you could buy a copy at Circuit City for only $49.99 which believe it or not was a great deal it was used extensively in education to teach people how to code but also used to build serious deskt

In [3]:
len(docs[0].page_content.split())

557

In [58]:
xx = "hi there my name is usman"
x = xx.split()
print(len(x))

6


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100, 
    length_function=len,
    )

docsvec = r_splitter.split_documents(docs)

In [18]:
docsvec

[Document(page_content='Pascal a procedural highlevel programming language famous for teaching a generation of kids from the 70s and 80s how to code it was created by Nicholas worth in the late 1960s and named after French mathematician blae Pascal it was originally based on the alol 60 language but expanded its data structuring abilities allowing developers to build Dynamic recursive data structures like trees and graphs it got its big break when it became the language of choice on the Apple 2 then Lisa and the Macintosh and eventually became the default highlevel language on nearly every PC over the years it evolved into a variety of other dialects most famously turbo Pascal brought to you by CP Creator Anders hilburg it was one of the first languages with its own full screen IDE and in 1983 you could buy a copy at Circuit City for only $49.99 which believe it or not was a great deal it was used extensively in education to teach people how to code but also used to build serious deskt

In [75]:
len(docsvec)

4

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
import os
from dotenv import load_dotenv
load_dotenv()
import pickle
import faiss
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
vectorstore_openAI = FAISS.from_documents(docsvec, embeddings)
vectorstore_openAI.save_local("vectorstore_openAI")

In [ ]:
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
llm

In [25]:
docsearch = FAISS.load_local("vectorstore_openAI",embeddings)

In [27]:
question = "who created pascal"
docsvec = docsearch.similarity_search(question,k = 2)
print(docsvec)

[Document(page_content='Pascal a procedural highlevel programming language famous for teaching a generation of kids from the 70s and 80s how to code it was created by Nicholas worth in the late 1960s and named after French mathematician blae Pascal it was originally based on the alol 60 language but expanded its data structuring abilities allowing developers to build Dynamic recursive data structures like trees and graphs it got its big break when it became the language of choice on the Apple 2 then Lisa and the Macintosh and eventually became the default highlevel language on nearly every PC over the years it evolved into a variety of other dialects most famously turbo Pascal brought to you by CP Creator Anders hilburg it was one of the first languages with its own full screen IDE and in 1983 you could buy a copy at Circuit City for only $49.99 which believe it or not was a great deal it was used extensively in education to teach people how to code but also used to build serious deskt

In [105]:
from langchain.memory import ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100,memory_key="chat_history",return_messages=True)

In [ ]:
memory

In [118]:
from langchain.prompts import PromptTemplate
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you its not  in the video, don't try to make up an answer.Keep the answer as concise as possible. Always say "thanks for asking!" on the next line at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

In [136]:
qa = "hi"

In [147]:
from langchain.chains import RetrievalQA
from langchain.chains import ConversationalRetrievalChain
retriever = vectorstore.as_retriever()
'''
qa = ConversationalRetrievalChain.from_chain_type(
    llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)
'''

qa_chain = RetrievalQA.from_chain_type(llm,
                                       retriever=retriever,
                                       #return_source_documents=True,
                                       chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
                                       memory = memory)

In [ ]:
qa_chain

In [139]:
qa

'hi'

In [ ]:
question = "which programming language is pascal like "
result = qa_chain({"query": question})

print(result)

In [ ]:
print(result)

In [ ]:
memory.load_memory_variables({})

In [ ]:
result["result"]